# Arquitectura OLAP en Spotify (Dataset Real 2024) 🎵
## Contexto:
Eres el Arquitecto de Datos Principal en una disquera. El equipo de ingeniería extrajo un dump transaccional masivo con las canciones más reproducidas de Spotify en 2024. El archivo plano es ancho, redundante y poco óptimo para análisis.

Tu misión es aplicar una tubería ETL para convertir este archivo crudo en un Esquema de Estrella en memoria, y posteriormente responder a 10 preguntas críticas del equipo de analítica utilizando el Cubo OLAP.

In [1]:
import pandas as pd
import numpy as np

# EXTRACT: Leemos directamente el archivo crudo desde el repositorio
url_dataset = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-III/datasets/Most%20Streamed%20Spotify%20Songs%202024.csv"

# Para evitar errores de codificación con caracteres especiales en nombres de artistas
spotify_raw = pd.read_csv(url_dataset, encoding='latin-1')

print("Muestra de datos crudos (OLTP):")
display(spotify_raw.head(3))
print("Columnas disponibles:", spotify_raw.columns.tolist())
display(spotify_raw)

Muestra de datos crudos (OLTP):


,Track,Album Name,Artist,Release Date,ISRC,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,...,SiriusXM Spins,Deezer Playlist Count,Deezer Playlist Reach,Amazon Playlist Count,Pandora Streams,Pandora Track Stations,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Explicit Track
0,MILLION DOLLAR BABY,Million Dollar Baby - Single,Tommy Richman,4/26/2024,QM24S2402528,1,725.4,"390,470,936","30,716","196,631,588",...,684,62.0,"17,598,718",114.0,"18,004,655","22,931","4,818,457","2,669,262",NaN,0
1,Not Like Us,Not Like Us,Kendrick Lamar,5/4/2024,USUG12400910,2,545.9,"323,703,884","28,113","174,597,137",...,3,67.0,"10,422,430",111.0,"7,780,028","28,444","6,623,075","1,118,279",NaN,1
2,i like the way you kiss me,I like the way you kiss me,Artemas,3/19/2024,QZJ842400387,3,538.4,"601,309,283","54,331","211,607,669",...,536,136.0,"36,321,847",172.0,"5,022,621","5,639","7,208,651","5,285,340",NaN,0


Columnas disponibles: ['Track', 'Album Name', 'Artist', 'Release Date', 'ISRC', 'All Time Rank', 'Track Score', 'Spotify Streams', 'Spotify Playlist Count', 'Spotify Playlist Reach', 'Spotify Popularity', 'YouTube Views', 'YouTube Likes', 'TikTok Posts', 'TikTok Likes', 'TikTok Views', 'YouTube Playlist Reach', 'Apple Music Playlist Count', 'AirPlay Spins', 'SiriusXM Spins', 'Deezer Playlist Count', 'Deezer Playlist Reach', 'Amazon Playlist Count', 'Pandora Streams', 'Pandora Track Stations', 'Soundcloud Streams', 'Shazam Counts', 'TIDAL Popularity', 'Explicit Track']


,Track,Album Name,Artist,Release Date,ISRC,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,...,SiriusXM Spins,Deezer Playlist Count,Deezer Playlist Reach,Amazon Playlist Count,Pandora Streams,Pandora Track Stations,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Explicit Track
0,MILLION DOLLAR BABY,Million Dollar Baby - Single,Tommy Richman,4/26/2024,QM24S2402528,1,725.4,"390,470,936","30,716","196,631,588",...,684,62.0,"17,598,718",114.0,"18,004,655","22,931","4,818,457","2,669,262",NaN,0
1,Not Like Us,Not Like Us,Kendrick Lamar,5/4/2024,USUG12400910,2,545.9,"323,703,884","28,113","174,597,137",...,3,67.0,"10,422,430",111.0,"7,780,028","28,444","6,623,075","1,118,279",NaN,1
2,i like the way you kiss me,I like the way you kiss me,Artemas,3/19/2024,QZJ842400387,3,538.4,"601,309,283","54,331","211,607,669",...,536,136.0,"36,321,847",172.0,"5,022,621","5,639","7,208,651","5,285,340",NaN,0
3,Flowers,Flowers - Single,Miley Cyrus,1/12/2023,USSM12209777,4,444.9,"2,031,280,633","269,802","136,569,078",...,"2,182",264.0,"24,684,248",210.0,"190,260,277","203,384",NaN,"11,822,942",NaN,0
4,Houdini,Houdini,Eminem,5/31/2024,USUG12403398,5,423.3,"107,034,922","7,223","151,469,874",...,1,82.0,"17,660,624",105.0,"4,493,884","7,006","207,179","457,017",NaN,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4595,For the Last Time,For the Last Time,$uicideboy$,9/5/2017,QM8DG1703420,"4,585",19.4,"305,049,963","65,770","5,103,054",...,NaN,2.0,"14,217",NaN,"20,104,066","13,184","50,633,006","656,337",NaN,1
4596,Dil Meri Na Sune,"Dil Meri Na Sune (From ""Genius"")",Atif Aslam,7/27/2018,INT101800122,"4,575",19.4,"52,282,360","4,602","1,449,767",...,NaN,1.0,927,NaN,NaN,NaN,NaN,"193,590",NaN,0
4597,Grace (feat. 42 Dugg),My Turn,Lil Baby,2/28/2020,USUG12000043,"4,571",19.4,"189,972,685","72,066","6,704,802",...,NaN,1.0,74,6.0,"84,426,740","28,999",NaN,"1,135,998",NaN,1
4598,Nashe Si Chadh Gayi,November Top 10 Songs,Arijit Singh,11/8/2016,INY091600067,"4,591",19.4,"145,467,020","14,037","7,387,064",...,NaN,NaN,NaN,7.0,"6,817,840",NaN,NaN,"448,292",NaN,0


### Fase 2: Transform & Load (Construyendo el Almacén)

Observa las columnas del dataset. Tienes texto (Track, Artist, Date) mezclado con métricas puras (Spotify Streams, YouTube Views). 

**Reto Arquitectónico:**
1. Crea una **Dimensión Tiempo (`dim_tiempo`)**: Extrae las columnas relacionadas al año, mes y día de lanzamiento. Elimina duplicados y crea un `id_tiempo`.
2. Crea una **Dimensión Pista (`dim_pista`)**: Extrae el nombre del track, el artista y si es explícita. Crea un `id_pista`.
3. Crea la **Tabla de Hechos (`fact_streams`)**: Cruza los IDs y conserva **únicamente** métricas numéricas (ej. Spotify Streams, YouTube Views, Track Score) junto con las llaves foráneas. NO debe haber texto aquí.

In [2]:
# --- ESCRIBE TU PIPELINE ETL AQUÍ ---

# 1. Crear dim_tiempo
dim_tiempo = spotify_raw[["Release Date"]].drop_duplicates().reset_index(drop=True)
dim_tiempo['id_tiempo'] = dim_tiempo.index + 1

display(dim_tiempo)

,Release Date,id_tiempo
0,4/26/2024,1
1,5/4/2024,2
2,3/19/2024,3
3,1/12/2023,4
4,5/31/2024,5
...,...,...
1557,4/9/2016,1558
1558,6/5/2023,1559
1559,10/31/2018,1560
1560,11/8/2016,1561


In [3]:
# 2. Crear dim_pista
# dim_pista = ...
dim_pista = spotify_raw[["Track"]].drop_duplicates().reset_index(drop=True)
dim_pista['track_id'] = dim_pista.index + 1

display(dim_pista)

,Track,track_id
0,MILLION DOLLAR BABY,1
1,Not Like Us,2
2,i like the way you kiss me,3
3,Flowers,4
4,Houdini,5
...,...,...
4365,For the Last Time,4366
4366,Dil Meri Na Sune,4367
4367,Grace (feat. 42 Dugg),4368
4368,Nashe Si Chadh Gayi,4369


In [4]:
# 3. Crear fact_streams
# fact_streams = ...

fact_streams = spotify_raw.merge(dim_tiempo, on="Release Date", how="left")
fact_streams = fact_streams.merge(dim_pista, on="Track", how="left")

fact_streams = fact_streams.drop(columns=["Track", "Album Name", "ISRC", "TIDAL Popularity"])
display(fact_streams)
print("Columnas disponibles:", fact_streams.columns.tolist())


,Artist,Release Date,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,Spotify Popularity,YouTube Views,YouTube Likes,...,Deezer Playlist Count,Deezer Playlist Reach,Amazon Playlist Count,Pandora Streams,Pandora Track Stations,Soundcloud Streams,Shazam Counts,Explicit Track,id_tiempo,track_id
0,Tommy Richman,4/26/2024,1,725.4,"390,470,936","30,716","196,631,588",92.0,"84,274,754","1,713,126",...,62.0,"17,598,718",114.0,"18,004,655","22,931","4,818,457","2,669,262",0,1,1
1,Kendrick Lamar,5/4/2024,2,545.9,"323,703,884","28,113","174,597,137",92.0,"116,347,040","3,486,739",...,67.0,"10,422,430",111.0,"7,780,028","28,444","6,623,075","1,118,279",1,2,2
2,Artemas,3/19/2024,3,538.4,"601,309,283","54,331","211,607,669",92.0,"122,599,116","2,228,730",...,136.0,"36,321,847",172.0,"5,022,621","5,639","7,208,651","5,285,340",0,3,3
3,Miley Cyrus,1/12/2023,4,444.9,"2,031,280,633","269,802","136,569,078",85.0,"1,096,100,899","10,629,796",...,264.0,"24,684,248",210.0,"190,260,277","203,384",NaN,"11,822,942",0,4,4
4,Eminem,5/31/2024,5,423.3,"107,034,922","7,223","151,469,874",88.0,"77,373,957","3,670,188",...,82.0,"17,660,624",105.0,"4,493,884","7,006","207,179","457,017",1,5,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4595,$uicideboy$,9/5/2017,"4,585",19.4,"305,049,963","65,770","5,103,054",71.0,"149,247,747","1,397,590",...,2.0,"14,217",NaN,"20,104,066","13,184","50,633,006","656,337",1,1497,4366
4596,Atif Aslam,7/27/2018,"4,575",19.4,"52,282,360","4,602","1,449,767",56.0,"943,920,245","5,347,766",...,1.0,927,NaN,NaN,NaN,NaN,"193,590",0,1030,4367
4597,Lil Baby,2/28/2020,"4,571",19.4,"189,972,685","72,066","6,704,802",65.0,"201,027,333","1,081,402",...,1.0,74,6.0,"84,426,740","28,999",NaN,"1,135,998",1,746,4368
4598,Arijit Singh,11/8/2016,"4,591",19.4,"145,467,020","14,037","7,387,064",66.0,"1,118,595,159","3,868,828",...,NaN,NaN,7.0,"6,817,840",NaN,NaN,"448,292",0,1561,4369


Columnas disponibles: ['Artist', 'Release Date', 'All Time Rank', 'Track Score', 'Spotify Streams', 'Spotify Playlist Count', 'Spotify Playlist Reach', 'Spotify Popularity', 'YouTube Views', 'YouTube Likes', 'TikTok Posts', 'TikTok Likes', 'TikTok Views', 'YouTube Playlist Reach', 'Apple Music Playlist Count', 'AirPlay Spins', 'SiriusXM Spins', 'Deezer Playlist Count', 'Deezer Playlist Reach', 'Amazon Playlist Count', 'Pandora Streams', 'Pandora Track Stations', 'Soundcloud Streams', 'Shazam Counts', 'Explicit Track', 'id_tiempo', 'track_id']


In [5]:
# 4. (LOAD) Reconstruir el Cubo en memoria para las consultas
cubo_olap = fact_streams 
cubo_olap["Release Date"] = pd.to_datetime(cubo_olap["Release Date"]) 
cubo_olap['Day'] = cubo_olap['Release Date'].dt.day
cubo_olap['Month'] = cubo_olap['Release Date'].dt.month
cubo_olap['Year'] = cubo_olap['Release Date'].dt.year
# YA FUE
display(cubo_olap)

,Artist,Release Date,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,Spotify Popularity,YouTube Views,YouTube Likes,...,Pandora Streams,Pandora Track Stations,Soundcloud Streams,Shazam Counts,Explicit Track,id_tiempo,track_id,Day,Month,Year
0,Tommy Richman,2024-04-26,1,725.4,"390,470,936","30,716","196,631,588",92.0,"84,274,754","1,713,126",...,"18,004,655","22,931","4,818,457","2,669,262",0,1,1,26,4,2024
1,Kendrick Lamar,2024-05-04,2,545.9,"323,703,884","28,113","174,597,137",92.0,"116,347,040","3,486,739",...,"7,780,028","28,444","6,623,075","1,118,279",1,2,2,4,5,2024
2,Artemas,2024-03-19,3,538.4,"601,309,283","54,331","211,607,669",92.0,"122,599,116","2,228,730",...,"5,022,621","5,639","7,208,651","5,285,340",0,3,3,19,3,2024
3,Miley Cyrus,2023-01-12,4,444.9,"2,031,280,633","269,802","136,569,078",85.0,"1,096,100,899","10,629,796",...,"190,260,277","203,384",NaN,"11,822,942",0,4,4,12,1,2023
4,Eminem,2024-05-31,5,423.3,"107,034,922","7,223","151,469,874",88.0,"77,373,957","3,670,188",...,"4,493,884","7,006","207,179","457,017",1,5,5,31,5,2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4595,$uicideboy$,2017-09-05,"4,585",19.4,"305,049,963","65,770","5,103,054",71.0,"149,247,747","1,397,590",...,"20,104,066","13,184","50,633,006","656,337",1,1497,4366,5,9,2017
4596,Atif Aslam,2018-07-27,"4,575",19.4,"52,282,360","4,602","1,449,767",56.0,"943,920,245","5,347,766",...,NaN,NaN,NaN,"193,590",0,1030,4367,27,7,2018
4597,Lil Baby,2020-02-28,"4,571",19.4,"189,972,685","72,066","6,704,802",65.0,"201,027,333","1,081,402",...,"84,426,740","28,999",NaN,"1,135,998",1,746,4368,28,2,2020
4598,Arijit Singh,2016-11-08,"4,591",19.4,"145,467,020","14,037","7,387,064",66.0,"1,118,595,159","3,868,828",...,"6,817,840",NaN,NaN,"448,292",0,1561,4369,8,11,2016


### Explica porqué solo hay 2 tablas dimensiones o si podrías incluir alguna otra. 
# Porque solo hay 2 tablas categóricas con importancia, los demás son ID's 

### Fase 3: Consultas Analíticas (10 Retos)

El equipo de analítica te ha enviado 10 peticiones. Para cada una, debes:
1. Escribir el código en Pandas usando la matriz `cubo_olap` que construiste.
2. Identificar y escribir en un comentario qué tipo de operación OLAP estás aplicando: **Roll-up, Drill-down, Slice o Dice**.

In [6]:
# RETO 1: El CEO quiere ver el total acumulado de 'Spotify Streams' agrupado por 'Artist'.
# Operación OLAP: [Escribe aquí si es Roll-up, Drill-down, Slice o Dice]

# ROLL-UP
#respuesta_1 = cubo_olap.groupby("Artist")["Spotify Streams"].sum()
#print(respuesta_1)

# RETO 2: El equipo de marketing quiere ver todas las métricas únicamente de las canciones lanzadas en el año 2024.
# Operación OLAP: SLICE

#respuesta_2 = cubo_olap[cubo_olap["Year"] == "2024" ] 
#display(respuesta_2)

# RETO 3: Finanzas requiere el total de 'Spotify Streams' desglosado primero por Año de Lanzamiento (Year) y luego detallado por Mes de Lanzamiento (Month).
# Operación OLAP: ROLL UP
#respuesta_3 = cubo_olap.groupby(["Year", "Month"])["Spotify Streams"].sum()
#display(respuesta_3)

# RETO 4: Auditoría te pide extraer el rendimiento de las canciones que son explícitas Y que además fueron lanzadas en el año 2024.
#Operación OLAP: DICE
#respuesta_4 = cubo_olap[(cubo_olap["Year"] == "2024") & (cubo_olap["Explicit Track"])]

# RETO 5: Queremos disminuir el nivel de detalle y ver solamente el promedio del 'Track Score' agrupado por Año de Lanzamiento.
# Operación OLAP:ROLL UP
#respuesta_5 = cubo_olap.groupby(["Year"])["Track Score"].mean()


In [ ]:
# RETO 6: Muestra el comportamiento analítico aislando únicamente los datos del artista "Bad Bunny".
# Operación OLAP: SLICE
#respuesta_6 = cubo_olap[cubo_olap["Artist"] == "Bad Bunny"]
#display(respuesta_6)

# RETO 7: Aumenta el nivel de detalle: Muestra el total de 'YouTube Views' agrupado por Artista, y dentro de cada Artista, desglosado por Nombre de la Canción (Track).
# Operación OLAP: DRILL DOWN
#respuesta_7 = cubo_olap.groupby(["Artist", "track_id"])["YouTube Views"].sum()
#display(respuesta_7)
# RETO 8: Extrae un sub-cubo que contenga únicamente canciones lanzadas en el Mes 12 (Diciembre) del Año 2023.
# Operación OLAP: DICE
#respuesta_8 =  cubo_olap[(cubo_olap["Month"] == 12) & (cubo_olap["Year"] == 2023)]
#display(respuesta_8)
# RETO 9: Resume la información calculando el total histórico de 'Spotify Streams' de todos los tiempos, agrupado por Mes (para saber en qué mes históricamente se lanza la música más exitosa).
# Operación OLAP: ROLL UP
#respuesta_9 = cubo_olap.groupby(["Month"])["Spotify Streams"].sum()
#display(respuesta_9)
# RETO 10: Extrae un bloque de información filtrando las canciones de "Taylor Swift" que NO sean explícitas y hayan salido en 2024.
# Operación OLAP: SLICE
#respuesta_10 = cubo_olap[(cubo_olap["Artist"] == "Taylor Swift") & (cubo_olap["Explicit Track"] == 0) & (cubo_olap["Year"] == "2024")] 
#display(respuesta_10)

,Artist,Release Date,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,Spotify Popularity,YouTube Views,YouTube Likes,...,Pandora Streams,Pandora Track Stations,Soundcloud Streams,Shazam Counts,Explicit Track,id_tiempo,track_id,Day,Month,Year
